# 05 Avaliação e Análises

**Métricas de avaliação** tabela comparativa entity-level (P/R/F1 por classe + micro/macro)
recomputada a partir dos modelos salvos em `models/`, para garantir consistência com o estado
atual do código (independente de `results/metrics.json` ter sido gerado por uma versão anterior).

**Análise qualitativa de erros** comparação Baseline 2 (DDsmall) vs. melhor variante de
Proposta 2 (DDsmall + pseudo), categorizando cada discrepância gold↔predito em:

| Categoria | Subcategoria | Corresponde a (enunciado) |
|---|---|---|
| `erro_classe` | — | erro de classe entre Pessoa/Local/Organização |
| `erro_fronteira` | `composta_parcial` | entidade composta reconhecida apenas parcialmente |
| `erro_fronteira` | `span_maior` / `deslocado` | erro de fronteira do span |
| `falso_negativo` | `variacao_ortografica` | falso negativo por variação ortográfica |
| `falso_negativo` | `fora_do_lexico` | entidade informal/apelido fora do léxico |
| `falso_negativo` | `no_lexico_nao_reconhecida` | entidade no léxico que o modelo não generalizou |
| `falso_positivo` | — | falso positivo por termo ambíguo |

Acertos exatos (mesmo span e mesma classe) não entram nessa lista só as discrepâncias.

In [1]:
import json
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
from src.data_loading import load_ddsmall
from src.evaluation import load_and_evaluate, NER_LABELS, PRETRAINED_LABEL_MAP

ROOT = Path('..')
RESULTS_DIR = ROOT / 'results'
test_recs = load_ddsmall(ROOT / 'data/raw/DDsmall/test/dd_corpus_small_test.json')

with open(ROOT / 'data/lexicon/lexicon_filtered.json', encoding='utf-8') as f:
    lexicon = json.load(f)

# Modelos disponíveis em disco caminho e label_map (None = labels já no esquema DDsmall)
MODEL_CONFIGS = [
    ('Baseline 1 (zero-shot)',   'pt_core_news_lg',              PRETRAINED_LABEL_MAP),
    ('Proposta 1 (EntityRuler)', ROOT / 'models/proposta1',      None),
    ('Baseline 2 (DDsmall)',     ROOT / 'models/baseline2',      None),
    ('Proposta 2 (20k pseudo)',  ROOT / 'models/proposta2_20000', None),
    ('Proposta 2 (completo)',    ROOT / 'models/proposta2_full', None),
]

all_scores, rows = {}, []
for name, path, label_map in MODEL_CONFIGS:
    if isinstance(path, Path) and not path.exists():
        print(f'[{name}] modelo não encontrado em {path} — pulando')
        continue
    scores = load_and_evaluate(path, test_recs, label_map)
    all_scores[name] = scores
    row = {'Modelo': name}
    for lbl in NER_LABELS:
        row[f'{lbl[:3]}_F1'] = scores.get(lbl, {}).get('f1', float('nan'))
    row['micro_F1'] = scores['micro']['f1']
    row['macro_F1'] = scores['macro']['f1']
    rows.append(row)

df = pd.DataFrame(rows).set_index('Modelo')
print(df.to_string(float_format=lambda v: f'{v:.4f}'))

RESULTS_DIR.mkdir(exist_ok=True)
with open(RESULTS_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(all_scores, f, ensure_ascii=False, indent=2)
print(f'\nMétricas salvas em {RESULTS_DIR / "metrics.json"}')

                          Per_F1  Loc_F1  Org_F1  micro_F1  macro_F1
Modelo                                                              
Baseline 1 (zero-shot)    0.4468  0.3752  0.1874    0.3619    0.3365
Proposta 1 (EntityRuler)  0.4677  0.6901  0.6786    0.6424    0.6121
Baseline 2 (DDsmall)      0.7353  0.8308  0.7494    0.7954    0.7718
Proposta 2 (20k pseudo)   0.5041  0.7285  0.6915    0.6763    0.6414
Proposta 2 (completo)     0.4565  0.6927  0.6792    0.6413    0.6095

Métricas salvas em ..\results\metrics.json


---
## Análise qualitativa de erros

Comparamos **Baseline 2** (DDsmall apenas) contra a melhor variante de **Proposta 2**
disponível (escolhida automaticamente pelo micro-F1), já que essa é a comparação
central do trabalho (pergunta principal do enunciado).

In [2]:
from src.evaluation import predict_records, categorize_errors, summarize_error_categories

# Escolhe a melhor variante de Proposta 2 pelo micro-F1 já calculado na célula anterior
proposta2_candidates = [n for n in all_scores if n.startswith('Proposta 2') ]
best_p2_name = max(proposta2_candidates, key=lambda n: all_scores[n]['micro']['f1'])
best_p2_path = dict((n, p) for n, p, _ in MODEL_CONFIGS)[best_p2_name]
print(f'Melhor variante de Proposta 2: {best_p2_name}  (micro-F1={all_scores[best_p2_name]["micro"]["f1"]:.4f})')

import spacy
nlp_b2 = spacy.load(ROOT / 'models/baseline2')
nlp_p2 = spacy.load(best_p2_path)

preds_b2 = predict_records(nlp_b2, test_recs)
preds_p2 = predict_records(nlp_p2, test_recs)

errors_b2 = categorize_errors(test_recs, preds_b2, lexicon=lexicon)
errors_p2 = categorize_errors(test_recs, preds_p2, lexicon=lexicon)

summary_b2 = summarize_error_categories(errors_b2)
summary_p2 = summarize_error_categories(errors_p2)

cat_table = pd.DataFrame({
    'Baseline 2': pd.Series({f'{c}/{s}' if s else c: v for (c, s), v in summary_b2.items()}),
    best_p2_name: pd.Series({f'{c}/{s}' if s else c: v for (c, s), v in summary_p2.items()}),
}).fillna(0).astype(int)
print(cat_table.to_string())

Melhor variante de Proposta 2: Proposta 2 (20k pseudo)  (micro-F1=0.6763)
                                          Baseline 2  Proposta 2 (20k pseudo)
erro_classe                                      119                       83
erro_fronteira/composta_parcial                  110                      217
erro_fronteira/deslocado                           3                        2
erro_fronteira/span_maior                        109                       52
falso_negativo/fora_do_lexico                    188                      557
falso_negativo/no_lexico_nao_reconhecida          42                       33
falso_negativo/variacao_ortografica               68                      219
falso_positivo                                   264                      296


### Amostras por categoria (inspeção manual)

Cada bloco abaixo mostra até 5 exemplos de uma categoria/subcategoria, com o relato
completo e o trecho gold vs. predito, para apoiar a discussão escrita no relatório final.

In [3]:
from src.evaluation import sample_errors

def show_samples(errors, category, subcategory=None, n=5, context=60):
    samples = sample_errors(errors, category, subcategory, n)
    label = f'{category}/{subcategory}' if subcategory else category
    print(f'\n=== {label} — {len(samples)} amostra(s) (de {sum(1 for e in errors if e["category"]==category and e.get("subcategory")==subcategory)} no total) ===')
    for e in samples:
        gold = e.get('gold')
        pred = e.get('pred')
        anchor = (gold or pred)
        start, end = max(0, anchor['start'] - context), anchor['end'] + context
        snippet = e['text'][start:end].replace('\n', ' ')
        print(f'  doc={e["doc_id"]}  ...{snippet}...')
        if gold:
            print(f'    GOLD : [{gold["label"]}] "{gold["surface"]}"')
        if pred:
            print(f'    PRED : [{pred["label"]}] "{pred["surface"]}"')
        if e.get('closest_lexicon_match'):
            print(f'    LEXICO mais próximo: "{e["closest_lexicon_match"]}"')

print('#'*20, 'PROPOSTA 2 —', best_p2_name, '#'*20)
show_samples(errors_p2, 'falso_positivo')
show_samples(errors_p2, 'falso_negativo', 'variacao_ortografica')
show_samples(errors_p2, 'falso_negativo', 'fora_do_lexico')
show_samples(errors_p2, 'falso_negativo', 'no_lexico_nao_reconhecida')
show_samples(errors_p2, 'erro_classe')
show_samples(errors_p2, 'erro_fronteira', 'composta_parcial')

print('\n' + '#'*20, 'BASELINE 2', '#'*20)
show_samples(errors_b2, 'falso_positivo')
show_samples(errors_b2, 'falso_negativo', 'fora_do_lexico')
show_samples(errors_b2, 'erro_classe')

#################### PROPOSTA 2 — Proposta 2 (20k pseudo) ####################

=== falso_positivo — 5 amostra(s) (de 296 no total) ===
  doc=2  ...s para entrarem na comunidade . Pede que seja tomada alguma providência ....
    PRED : [Location] "providência"
  doc=5  ...dos . fizemos orações e nada adiantou . Pois tomem logo uma providência vcs da disque denuncia . Eles já estão levando muito a séri...
    PRED : [Location] "providência"
  doc=9  ...Os bandidos estão escondendo os carros na polimix os funcionários são obrigados a abrir o portão para esconde...
    PRED : [Organization] "polimix"
  doc=13  ...es começam a vender as 10 da noite onze em diante eu até vi polícias aqui durante o dia mas a venda é pela madruga a dentro é um...
    PRED : [Organization] "polícias"
  doc=18  ... placa MYE4590 encontra se abandonada na antiga estrada rio são Paulo km 32 nova Iguaçu em frente à loja Max casa de fogos km 32 ...
    PRED : [Location] "são Paulo"

=== falso_negativo/variacao_orto

In [4]:
import json as _json

ERR_DIR = RESULTS_DIR / 'error_analysis'
ERR_DIR.mkdir(parents=True, exist_ok=True)

with open(ERR_DIR / 'baseline2_errors.json', 'w', encoding='utf-8') as f:
    _json.dump(errors_b2, f, ensure_ascii=False, indent=2)
with open(ERR_DIR / f'proposta2_errors.json', 'w', encoding='utf-8') as f:
    _json.dump(errors_p2, f, ensure_ascii=False, indent=2)
with open(ERR_DIR / 'category_summary.json', 'w', encoding='utf-8') as f:
    _json.dump({
        'baseline2': {f'{c}/{s}' if s else c: v for (c, s), v in summary_b2.items()},
        best_p2_name: {f'{c}/{s}' if s else c: v for (c, s), v in summary_p2.items()},
    }, f, ensure_ascii=False, indent=2)

print(f'Erros categorizados salvos em {ERR_DIR}/')

Erros categorizados salvos em ..\results\error_analysis/
